In [ ]:
#@title Ячейка 0 — Drive, BASE, зависимости, GPU
import sys, os, importlib
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/rag_exp"
if BASE not in sys.path:
    sys.path.insert(0, BASE)
importlib.invalidate_caches()

os.environ.pop("HF_HOME", None)
os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs("/content/hf_cache", exist_ok=True)
os.makedirs(f"{BASE}/results", exist_ok=True)

import torch
assert torch.cuda.is_available(), "GPU не подключён! Среда → Сменить → L4 GPU"
print(torch.cuda.get_device_name(0),
      f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("BASE =", BASE)


Mounted at /content/drive
NVIDIA L4 | VRAM 23.7 GB
BASE = /content/drive/MyDrive/rag_exp


In [ ]:
#@title Ячейка 1 — зависимости (sentence-transformers нужен для e5/bge/MiniLM)
!pip install -q -U "transformers>=4.51.0" "sentence-transformers>=3.0" bitsandbytes accelerate
!pip install -q scikit-learn pandas rank_bm25 pymupdf scipy
print("deps ok")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 117.3 MB/s eta 0:00:00
deps ok


In [ ]:
#@title Ячейка 2 — rag_common + init (БЕЗ загрузки модели — модели грузим в цикле 2B)
import sys, importlib
sys.path.append(BASE)
importlib.invalidate_caches()
import rag_common; importlib.reload(rag_common)
from rag_common import (init_common, build_or_load_index, SparseIndex,
                        run_search, dense_search, hybrid_rrf_search, sparse_search,
                        precision_at_k, recall_at_k, mrr, ndcg_at_k, hit_at_k, average_precision)

# init без реальной модели — модель/токенайзер подставим в цикле 2B
init_common(BASE, None, None, None, None, None)
rag_common.BASE        = BASE
rag_common.INDEX_CACHE = f"{BASE}/index_cache"

import inspect
print("rag_common ok | BASE:", rag_common.BASE, "| INDEX_CACHE:", rag_common.INDEX_CACHE)
print("дедуп _ids_from:", "seen" in inspect.getsource(rag_common._ids_from))


rag_common init: MODEL=None, INDEX_CACHE=/content/drive/MyDrive/rag_exp/index_cache, reranker=—, oai=—
rag_common ok | BASE: /content/drive/MyDrive/rag_exp | INDEX_CACHE: /content/drive/MyDrive/rag_exp/index_cache
дедуп _ids_from: True


In [ ]:
#@title Ячейка 3 — чтение PDF корпуса → real_docs
import fitz, os
from doc_map import FILE_TO_ID
CORPUS_DIR = f"{BASE}/corpus_phase1"
real_docs = []
for fname, doc_id in FILE_TO_ID.items():
    pdf = fitz.open(os.path.join(CORPUS_DIR, fname))
    real_docs.append({"id": doc_id, "text": "\n".join(p.get_text() for p in pdf)})
    pdf.close()
print("Документов:", len(real_docs), "| символов:", sum(len(d['text']) for d in real_docs))


Документов: 11 | символов: 3444362


In [ ]:
#@title Ячейка 4 — QA_REAL (чанковая разметка 30/44, как в rag_03)
import re, pandas as pd
df_reg = pd.read_csv(f"{BASE}/qa_registry.csv")
good = df_reg[df_reg["статус"].isin(["GOOD","GOOD_SYNTHETIC","GOOD_OTHER_ED"])].copy()
qcol = "вопрос_полный" if "вопрос_полный" in df_reg.columns else "вопрос"

# индекс на chunk_size=1024 нужен для разметки чанков (chunk_id зависит от размера!)
# модели ещё нет → строим разметку по тексту чанков, без эмбеддинга:
def chunk_text_local(text, size, overlap):
    return rag_common.chunk_text(text, size, overlap)

# собираем чанки 1024 чисто текстово (без векторов) для маппинга пункт→chunk_id
all_chunks, chunk_meta = [], []
for d in real_docs:
    for j, ch in enumerate(chunk_text_local(d["text"], 1024, 0.1)):
        all_chunks.append(ch); chunk_meta.append({"doc_id": d["id"], "chunk_idx": j})

def norm(s): return re.sub(r"\s+", " ", str(s).lower()).strip()
nchunks = [norm(c) for c in all_chunks]
by_doc = {}
for i, m in enumerate(chunk_meta):
    by_doc.setdefault(m["doc_id"], []).append(i)

def chunk_from_note(note):
    if pd.isna(note): return None
    m = re.search(r"чанк[а-я]*\s*:?\s*(\d+)", str(note), re.I)
    return int(m.group(1)) if m else None
def clause_from_note(note):
    if pd.isna(note): return None
    m = re.search(r"п\.?\s*(\d+(?:\.\d+)+)", str(note)); return m.group(1) if m else None
def first_clause(clause):
    if pd.isna(clause): return None
    nums = re.findall(r"\d+(?:\.\d+)+", str(clause)); return nums[0] if nums else None
def is_comparative(clause, question):
    s = f"{clause} {question}".lower()
    return (" vs " in str(clause).lower()) or ("отлич" in s) or ("различ" in s)
def chunk_from_clause(doc_id, clause):
    if clause is None or doc_id not in by_doc: return None
    rx = re.compile(re.escape(clause.strip()).replace(r"\.", r"\.\s?"))
    for i in by_doc[doc_id]:
        if rx.search(nchunks[i]): return i
    return None

QA_REAL, cov_note, cov_clause, cov_doc = [], [], [], []
for _, row in good.iterrows():
    qid, doc_id = int(row["Q"]), row["doc_id"]
    q = {"query": str(row[qcol]), "Q": qid, "категория": row["категория"], "doc_id": doc_id}
    gold = chunk_from_note(row.get("примечание"))
    if gold is not None: cov_note.append(qid)
    elif is_comparative(row.get("пункт"), row[qcol]): gold = None
    else:
        clause = first_clause(row.get("пункт")) or clause_from_note(row.get("примечание"))
        if clause and clause.count(".") >= 2:
            gold = chunk_from_clause(doc_id, clause)
            if gold is not None: cov_clause.append(qid)
    if gold is not None: q["relevant"], q["level"] = {gold: 3}, "chunk"
    else: q["relevant"], q["level"] = {doc_id: 3}, "doc"; cov_doc.append(qid)
    QA_REAL.append(q)

print(f"QA_REAL: {len(QA_REAL)} | чанк-примечание {len(cov_note)}, чанк-пункт {len(cov_clause)}, документный {len(cov_doc)}")
print(f"на уровне чанков: {len(cov_note)+len(cov_clause)} из {len(QA_REAL)}")


QA_REAL: 44 | чанк-примечание 5, чанк-пункт 25, документный 14
на уровне чанков: 30 из 44


In [ ]:
#@title Ячейка 5 — реестр моделей 2B + функции encode под каждое семейство
import torch, time, gc
import torch.nn.functional as F
import rag_common

# --- РЕЕСТР: матрица 2B (без e5-стиля для Qwen, см. методику) ---
MODEL_REGISTRY = [
    # run, имя HF,                              семейство,  dim,  промпт
    ("2B.1", "Qwen/Qwen3-Embedding-4B",        "qwen",     2560, "none"),
    ("2B.2", "Qwen/Qwen3-Embedding-4B",        "qwen",     2560, "instruct"),
    ("2B.3", "Qwen/Qwen3-Embedding-8B",        "qwen",     4096, "none"),
    ("2B.4", "Qwen/Qwen3-Embedding-8B",        "qwen",     4096, "instruct"),
    ("2B.5", "intfloat/multilingual-e5-large", "e5",       1024, "e5"),
    ("2B.6", "BAAI/bge-m3",                    "st",       1024, "none"),
    ("2B.7", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "st", 384, "none"),
]

_loaded = {"name": None, "obj": None, "tok": None, "family": None}

def unload_model():
    """Выгрузить текущую модель из VRAM."""
    for k in ("obj", "tok"):
        if _loaded[k] is not None:
            del _loaded[k]; _loaded[k] = None
    _loaded["name"] = None; _loaded["family"] = None
    gc.collect(); torch.cuda.empty_cache()
    print(f"  [выгружено] свободно VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} ГБ")

def load_model(hf_name, family):
    """Загрузить модель нужного семейства, выгрузив предыдущую."""
    if _loaded["name"] == hf_name:
        return
    unload_model()
    print(f"  [загрузка] {hf_name} ({family}) ...")
    t0 = time.time()
    if family == "qwen":
        from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
        tok = AutoTokenizer.from_pretrained(hf_name, padding_side="left")
        bnb = BitsAndBytesConfig(load_in_8bit=True)
        obj = AutoModel.from_pretrained(hf_name, quantization_config=bnb, device_map="auto").eval()
        _loaded.update(obj=obj, tok=tok)
    else:  # e5 / st (bge-m3, MiniLM) — через SentenceTransformer
        from sentence_transformers import SentenceTransformer
        obj = SentenceTransformer(hf_name, device="cuda")
        _loaded.update(obj=obj, tok=None)
    _loaded.update(name=hf_name, family=family)
    print(f"  [готово] за {time.time()-t0:.0f} с | VRAM занято: {torch.cuda.memory_allocated()/1e9:.1f} ГБ")

# --- ENCODE под каждое семейство (подменяет rag_common.encode) ---
@torch.no_grad()
def _encode_qwen(texts, dim=None, max_length=512, batch_size=16):
    out, tok, model = [], _loaded["tok"], _loaded["obj"]
    for i in range(0, len(texts), batch_size):
        b = texts[i:i+batch_size]
        enc = tok(b, padding=True, truncation=True, max_length=max_length,
                  return_tensors="pt").to(model.device)
        h = model(**enc).last_hidden_state
        v = rag_common.last_token_pool(h, enc["attention_mask"])
        if dim: v = v[:, :dim]
        v = F.normalize(v, p=2, dim=1)
        out.append(v.float().cpu()); del enc, h, v; torch.cuda.empty_cache()
    return torch.cat(out, 0)

@torch.no_grad()
def _encode_st(texts, dim=None, max_length=512, batch_size=32):
    # bge-m3 / MiniLM: SentenceTransformer сам пулит и нормализует
    v = _loaded["obj"].encode(texts, batch_size=batch_size, convert_to_numpy=True,
                              normalize_embeddings=True, show_progress_bar=False)
    return torch.tensor(v)

@torch.no_grad()
def _encode_e5(texts, dim=None, max_length=512, batch_size=32):
    # e5: префикс passage: ко всему (запрос префиксуем отдельно, см. ниже)
    pref = [f"passage: {t}" for t in texts]
    v = _loaded["obj"].encode(pref, batch_size=batch_size, convert_to_numpy=True,
                              normalize_embeddings=True, show_progress_bar=False)
    return torch.tensor(v)

print("реестр готов:", len(MODEL_REGISTRY), "runs")
for r in MODEL_REGISTRY:
    print(" ", r[0], r[1].split("/")[-1], "| dim", r[3], "| промпт:", r[4])


реестр готов: 7 runs
  2B.1 Qwen3-Embedding-4B | dim 2560 | промпт: none
  2B.2 Qwen3-Embedding-4B | dim 2560 | промпт: instruct
  2B.3 Qwen3-Embedding-8B | dim 4096 | промпт: none
  2B.4 Qwen3-Embedding-8B | dim 4096 | промпт: instruct
  2B.5 multilingual-e5-large | dim 1024 | промпт: e5
  2B.6 bge-m3 | dim 1024 | промпт: none
  2B.7 paraphrase-multilingual-MiniLM-L12-v2 | dim 384 | промпт: none


In [ ]:
#@title Ячейка 6 — индекс под промпт + патч эмбеддинга запроса
import rag_common, numpy as np, json, os, time

def _query_prefix(prompt_mode, query):
    """Префикс к ЗАПРОСУ в зависимости от промпт-режима."""
    if prompt_mode == "none":      return query
    if prompt_mode == "instruct":  return rag_common.get_detailed_instruct(rag_common.SEARCH_TASK, query)
    if prompt_mode == "e5":        return f"query: {query}"
    return query

def _chunk_prefix(prompt_mode, text):
    """Префикс к ЧАНКУ в зависимости от промпт-режима."""
    if prompt_mode == "e5":        return f"passage: {text}"
    # qwen none/instruct и bge/minilm none — чанки без префикса
    return text

# Глобальный текущий промпт-режим (читается патченным dense_search)
CURRENT_PROMPT = {"mode": "none"}

# Патчим dense_search: вместо жёсткой Qwen-инструкции — префикс по текущему режиму
_orig_dense = rag_common.dense_search
def _dense_search_prompted(query, index, top_k=10, dim=2048):
    q_text = _query_prefix(CURRENT_PROMPT["mode"], query)
    q_vec = rag_common.encode([q_text], dim=dim).numpy()[0]
    sims = index["vectors"] @ q_vec
    order = np.argsort(-sims)[:top_k]
    return [{"rank": r, "score": float(sims[i]), "chunk_id": int(i),
             "doc_id": index["chunk_meta"][i]["doc_id"], "text": index["chunks"][i]}
            for r, i in enumerate(order, 1)]
rag_common.dense_search = _dense_search_prompted
print("dense_search пропатчен (префикс запроса по CURRENT_PROMPT)")

def build_index_2b(run, hf_name, family, dim, prompt_mode, docs, chunk_size=1024, overlap=0.1):
    """Строит/грузит индекс под конкретную модель+промпт. Ключ включает run (изоляция кэша)."""
    # ключ: модель + dim + промпт-режим (чтобы e5-passage не смешался с none)
    rag_common.MODEL = f"{hf_name}|{prompt_mode}"
    key = rag_common.index_key([d["id"] for d in docs], "fixed_512", chunk_size, overlap,
                               rag_common.MODEL, dim, "q8")
    path = f"{rag_common.INDEX_CACHE}/{key}.npz"; meta_path = f"{rag_common.INDEX_CACHE}/{key}.json"
    if os.path.exists(path):
        data = np.load(path, allow_pickle=True); meta = json.load(open(meta_path, encoding="utf-8"))
        print(f"  [cache HIT] {key}: {len(meta['chunks'])} чанков")
        return {"key": key, "vectors": data["vectors"], "chunks": meta["chunks"], "chunk_meta": meta["chunk_meta"]}
    print(f"  [cache MISS] строю {key} ...")
    t0 = time.time()
    all_chunks, chunk_meta = [], []
    for d in docs:
        for j, ch in enumerate(rag_common.chunk_text(d["text"], chunk_size, overlap)):
            all_chunks.append(ch); chunk_meta.append({"doc_id": d["id"], "chunk_idx": j})
    # к чанкам — префикс по промпт-режиму (для e5: passage:)
    to_embed = [_chunk_prefix(prompt_mode, c) for c in all_chunks]
    vectors = rag_common.encode(to_embed, dim=dim).numpy()
    np.savez_compressed(path, vectors=vectors)
    json.dump({"chunks": all_chunks, "chunk_meta": chunk_meta},
              open(meta_path, "w", encoding="utf-8"), ensure_ascii=False)
    print(f"  [built] {len(all_chunks)} чанков, {vectors.shape}, {time.time()-t0:.0f} с | индекс_время={time.time()-t0:.1f}")
    return {"key": key, "vectors": vectors, "chunks": all_chunks, "chunk_meta": chunk_meta, "build_sec": time.time()-t0}

print("build_index_2b готов")


dense_search пропатчен (префикс запроса по CURRENT_PROMPT)
build_index_2b готов


In [ ]:
#@title Ячейка 6.1 — дымовой тест 2B на MiniLM
run = MODEL_REGISTRY[-1]   # 2B.7 MiniLM
run_id, hf_name, family, dim, prompt = run
print("ТЕСТ:", run_id, "|", hf_name, "| dim", dim, "| промпт", prompt)

load_model(hf_name, family)
rag_common.encode = _encode_st            # ← ДОБАВЛЕНО: encode под ST-семейство
CURRENT_PROMPT["mode"] = prompt

idx = build_index_2b(run, hf_name, family, dim, prompt, real_docs)
sparse = SparseIndex(idx)
print("ключ индекса:", idx["key"], "| чанков:", len(idx["chunks"]))

q = next(x for x in QA_REAL if x["Q"] == 2)
ranked = rag_common.run_search("S6", q["query"], idx,
                               sparse_index=sparse, top_k=10, dim=dim, by="chunk")
print("ranked[:5]:", ranked[:5])
print("тип элемента ranked:", type(ranked[0]).__name__)

unload_model()





ТЕСТ: 2B.7 | sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 | dim 384 | промпт none
  [cache HIT] 92f5c89ab6c5fd3b: 3745 чанков
ключ индекса: 92f5c89ab6c5fd3b | чанков: 3745
ranked[:5]: [1338, 2333, 2780, 2335, 2680]
тип элемента ranked: int
  [выгружено] свободно VRAM: 23.4 ГБ


In [ ]:
#@title Ячейка 7 — ГЛАВНЫЙ ЦИКЛ 2B (doc-уровень через chunk_meta) → phase2b_models.csv
import pandas as pd, time, torch

# карта: семейство → функция encode
ENCODERS = {"qwen": _encode_qwen, "e5": _encode_e5, "st": _encode_st}

def eval_run_2b(run, docs, qa_set):
    run_id, hf_name, family, dim, prompt = run
    print(f"\n=== {run_id}: {hf_name} | dim {dim} | prompt {prompt} ===")
    load_model(hf_name, family)
    rag_common.encode = ENCODERS[family]      # ← КЛЮЧЕВОЕ: encode под семейство
    CURRENT_PROMPT["mode"] = prompt
    idx = build_index_2b(run, hf_name, family, dim, prompt, docs)
    sparse = SparseIndex(idx)
    meta = idx["chunk_meta"]                   # [{doc_id, chunk_idx}] по позициям

    m = {k: 0.0 for k in ["P@5","R@5","MRR","NDCG@5","Hit@5","MAP"]}
    t0 = time.time()
    for q in qa_set:
        rel_doc = {q["doc_id"]: 3}              # оцениваем по документу-источнику
        ranked_pos = rag_common.run_search("S6", q["query"], idx,
                                            sparse_index=sparse, top_k=10,
                                            dim=dim, by="chunk")   # позиции чанков
        doc_ranked, seen = [], set()
        for p in ranked_pos:
            pi = p if isinstance(p, int) else (p.get("chunk_id") if isinstance(p, dict) else None)
            if pi is None or pi >= len(meta):
                continue
            d = meta[pi]["doc_id"]
            if d not in seen:
                seen.add(d); doc_ranked.append(d)
        m["P@5"]    += rag_common.precision_at_k(doc_ranked, rel_doc, 5)
        m["R@5"]    += rag_common.recall_at_k(doc_ranked, rel_doc, 5)
        m["MRR"]    += rag_common.mrr(doc_ranked, rel_doc)
        m["NDCG@5"] += rag_common.ndcg_at_k(doc_ranked, rel_doc, 5)
        m["Hit@5"]  += rag_common.hit_at_k(doc_ranked, rel_doc, 5)
        m["MAP"]    += rag_common.average_precision(doc_ranked, rel_doc)

    n = len(qa_set)
    row = {"run": run_id, "model": hf_name.split("/")[-1], "family": family,
           "dim": dim, "prompt": prompt,
           **{k: round(v/n, 4) for k, v in m.items()},
           "search_sec": round(time.time()-t0, 2), "n_chunks": len(idx["chunks"])}
    unload_model()
    return row

rows = []
for run in MODEL_REGISTRY:
    try:
        row = eval_run_2b(run, real_docs, QA_REAL)
        rows.append(row)
        print(f"  ✅ R@5={row['R@5']} NDCG@5={row['NDCG@5']} Hit@5={row['Hit@5']} MRR={row['MRR']} | {row['search_sec']}s")
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"  ⚠️ {run[0]} упал: {type(e).__name__}: {e}")
        unload_model()

df_2b = pd.DataFrame(rows)
df_2b.to_csv(f"{BASE}/results/phase2b_models.csv", index=False)
print("\n=== ИТОГ 2B ===")
print(df_2b[["run","model","dim","prompt","NDCG@5","MRR","Hit@5","R@5","search_sec"]].to_string(index=False))




=== 2B.1: Qwen/Qwen3-Embedding-4B | dim 2560 | prompt none ===
  [выгружено] свободно VRAM: 23.4 ГБ
  [загрузка] Qwen/Qwen3-Embedding-4B (qwen) ...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

  [готово] за 21 с | VRAM занято: 2.3 ГБ
  [cache MISS] строю 374718cb10bd8c35 ...
  [built] 3745 чанков, (3745, 1024), 113 с | индекс_время=113.1
  [выгружено] свободно VRAM: 23.4 ГБ
  ✅ R@5=0.9773 NDCG@5=0.9072 Hit@5=0.9773 MRR=0.8826 | 1.96s

=== 2B.7: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 | dim 384 | prompt none ===
  [выгружено] свободно VRAM: 23.4 ГБ
  [загрузка] sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 (st) ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [готово] за 8 с | VRAM занято: 0.5 ГБ
  [cache HIT] 92f5c89ab6c5fd3b: 3745 чанков
  [выгружено] свободно VRAM: 23.4 ГБ
  ✅ R@5=0.9545 NDCG@5=0.7625 Hit@5=0.9545 MRR=0.6951 | 1.34s

=== ИТОГ 2B ===
 run                                 model  dim   prompt  NDCG@5    MRR  Hit@5    R@5  search_sec
2B.1                    Qwen3-Embedding-4B 2560     none  0.9254 0.8996 1.0000 1.0000       13.20
2B.2                    Qwen3-Embedding-4B 2560 instruct  0.9086 0.8769 1.0000 1.0000       13.10
2B.3                    Qwen3-Embedding-8B 4096     none  0.9367 0.9148 1.0000 1.0000       12.74
2B.4                    Qwen3-Embedding-8B 4096 instruct  0.9299 0.9053 1.0000 1.0000       13.05
2B.5                 multilingual-e5-large 1024       e5  0.9294 0.9129 0.9773 0.9773        1.90
2B.6                                bge-m3 1024     none  0.9072 0.8826 0.9773 0.9773        1.96
2B.7 paraphrase-multilingual-MiniLM-L12-v2  384     none  0.7625 0.6951 0.9545 0.9545        1.34


In [ ]:
#@title Ячейка 8 — статтесты топ-моделей 2B (Wilcoxon по MRR, по-вопросно)
import numpy as np
from scipy.stats import wilcoxon

# кандидаты для сравнения (run, hf_name, family, dim, prompt)
CANDIDATES = {
    "8B-none": ("2B.3", "Qwen/Qwen3-Embedding-8B", "qwen", 4096, "none"),
    "4B-none": ("2B.1", "Qwen/Qwen3-Embedding-4B", "qwen", 2560, "none"),
    "e5":      ("2B.5", "intfloat/multilingual-e5-large", "e5", 1024, "e5"),
}

def per_question_mrr(run, docs, qa_set):
    run_id, hf_name, family, dim, prompt = run
    load_model(hf_name, family)
    rag_common.encode = ENCODERS[family]
    CURRENT_PROMPT["mode"] = prompt
    idx = build_index_2b(run, hf_name, family, dim, prompt, docs)
    sparse = SparseIndex(idx); meta = idx["chunk_meta"]
    mrrs = []
    for q in qa_set:
        rel_doc = {q["doc_id"]: 3}
        pos = rag_common.run_search("S6", q["query"], idx, sparse_index=sparse,
                                    top_k=10, dim=dim, by="chunk")
        doc_ranked, seen = [], set()
        for p in pos:
            pi = p if isinstance(p, int) else None
            if pi is None or pi >= len(meta): continue
            d = meta[pi]["doc_id"]
            if d not in seen: seen.add(d); doc_ranked.append(d)
        mrrs.append(rag_common.mrr(doc_ranked, rel_doc))
    unload_model()
    return np.array(mrrs)

scores = {name: per_question_mrr(run, real_docs, QA_REAL) for name, run in CANDIDATES.items()}

print("\n=== средний MRR ===")
for name, s in scores.items():
    print(f"  {name}: {s.mean():.4f}")

print("\n=== Wilcoxon (парный, по MRR) ===")
for a, b in [("8B-none","4B-none"), ("8B-none","e5"), ("4B-none","e5")]:
    diff = scores[a] - scores[b]
    nz = np.sum(diff != 0)
    if nz == 0:
        print(f"  {a} vs {b}: идентичны (0 различий)")
    else:
        stat, p = wilcoxon(scores[a], scores[b])
        verdict = "ЗНАЧИМО" if p < 0.05 else "не значимо"
        print(f"  {a} vs {b}: p={p:.4f} ({verdict}), различий={nz}")


  [выгружено] свободно VRAM: 23.4 ГБ
  [загрузка] Qwen/Qwen3-Embedding-8B (qwen) ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [готово] за 23 с | VRAM занято: 8.2 ГБ
  [cache HIT] 21903bd59e5c05c0: 3745 чанков


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  [выгружено] свободно VRAM: 23.4 ГБ
  [выгружено] свободно VRAM: 23.4 ГБ
  [загрузка] Qwen/Qwen3-Embedding-4B (qwen) ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [готово] за 12 с | VRAM занято: 4.4 ГБ
  [cache HIT] 49d8988205626d3d: 3745 чанков
  [выгружено] свободно VRAM: 23.4 ГБ
  [выгружено] свободно VRAM: 23.4 ГБ
  [загрузка] intfloat/multilingual-e5-large (e5) ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  [готово] за 9 с | VRAM занято: 2.2 ГБ
  [cache HIT] 7962bbc94260d3bf: 3745 чанков
  [выгружено] свободно VRAM: 23.4 ГБ

=== средний MRR ===
  8B-none: 0.9148
  4B-none: 0.8996
  e5: 0.9129

=== Wilcoxon (парный, по MRR) ===
  8B-none vs 4B-none: p=0.4497 (не значимо), различий=4
  8B-none vs e5: p=0.8608 (не значимо), различий=7
  4B-none vs e5: p=0.4982 (не значимо), различий=5


In [ ]:
#@title Ячейка 9 — MRL-подгруппа: усечение 8B (4096 → 2048 → 1536) → дозапись в phase2b_models.csv
import pandas as pd, numpy as np, time

# исходный run 8B/4096 (его кэш уже построен)
SRC_RUN = ("2B.3", "Qwen/Qwen3-Embedding-8B", "qwen", 4096, "none")

def eval_truncated(src_run, target_dim, run_id, docs, qa_set):
    _, hf_name, family, full_dim, prompt = src_run
    # грузим готовый 4096-индекс из кэша (без модели — эмбеддинги уже есть)
    rag_common.MODEL = f"{hf_name}|{prompt}"
    key = rag_common.index_key([d["id"] for d in docs], "fixed_512", 1024, 0.1,
                               rag_common.MODEL, full_dim, "q8")
    import os, json
    data = np.load(f"{rag_common.INDEX_CACHE}/{key}.npz", allow_pickle=True)
    meta_j = json.load(open(f"{rag_common.INDEX_CACHE}/{key}.json", encoding="utf-8"))
    full_vecs = data["vectors"]                       # (3745, 4096), уже нормированы
    chunks, chunk_meta = meta_j["chunks"], meta_j["chunk_meta"]

    # MRL-срез: берём первые target_dim координат и ПЕРЕНОРМИРУЕМ
    sub = full_vecs[:, :target_dim].astype(np.float32)
    sub = sub / (np.linalg.norm(sub, axis=1, keepdims=True) + 1e-12)
    idx = {"key": f"{key}_mrl{target_dim}", "vectors": sub,
           "chunks": chunks, "chunk_meta": chunk_meta}
    sparse = SparseIndex(idx); m_meta = idx["chunk_meta"]

    # для поиска нужна модель (запрос кодируется), encode под Qwen, срез запроса до target_dim
    load_model(hf_name, family)
    rag_common.encode = ENCODERS[family]
    CURRENT_PROMPT["mode"] = prompt

    m = {k: 0.0 for k in ["P@5","R@5","MRR","NDCG@5","Hit@5","MAP"]}
    t0 = time.time()
    for q in qa_set:
        rel_doc = {q["doc_id"]: 3}
        pos = rag_common.run_search("S6", q["query"], idx, sparse_index=sparse,
                                    top_k=10, dim=target_dim, by="chunk")
        doc_ranked, seen = [], set()
        for p in pos:
            pi = p if isinstance(p, int) else None
            if pi is None or pi >= len(m_meta): continue
            d = m_meta[pi]["doc_id"]
            if d not in seen: seen.add(d); doc_ranked.append(d)
        m["P@5"]    += rag_common.precision_at_k(doc_ranked, rel_doc, 5)
        m["R@5"]    += rag_common.recall_at_k(doc_ranked, rel_doc, 5)
        m["MRR"]    += rag_common.mrr(doc_ranked, rel_doc)
        m["NDCG@5"] += rag_common.ndcg_at_k(doc_ranked, rel_doc, 5)
        m["Hit@5"]  += rag_common.hit_at_k(doc_ranked, rel_doc, 5)
        m["MAP"]    += rag_common.average_precision(doc_ranked, rel_doc)
    n = len(qa_set)
    row = {"run": run_id, "model": hf_name.split("/")[-1], "family": family,
           "dim": target_dim, "prompt": prompt + "+mrl",
           **{k: round(v/n, 4) for k, v in m.items()},
           "search_sec": round(time.time()-t0, 2), "n_chunks": len(chunks)}
    unload_model()
    return row

mrl_rows = []
for tgt, rid in [(2048, "2B.8"), (1536, "2B.9")]:
    print(f"\n=== {rid}: 8B усечённый до dim {tgt} ===")
    try:
        row = eval_truncated(SRC_RUN, tgt, rid, real_docs, QA_REAL)
        mrl_rows.append(row)
        print(f"  ✅ R@5={row['R@5']} NDCG@5={row['NDCG@5']} Hit@5={row['Hit@5']} MRR={row['MRR']}")
    except Exception as e:
        import traceback; traceback.print_exc()
        unload_model()

# дозапись в общий CSV
df_mrl = pd.DataFrame(mrl_rows)
df_all = pd.concat([pd.read_csv(f"{BASE}/results/phase2b_models.csv"), df_mrl], ignore_index=True)
df_all.to_csv(f"{BASE}/results/phase2b_models.csv", index=False)

# сравнение с полным 4096
print("\n=== MRL: деградация 8B при усечении ===")
full = pd.read_csv(f"{BASE}/results/phase2b_models.csv")
view = full[full["run"].isin(["2B.3","2B.8","2B.9"])][["run","dim","NDCG@5","MRR","Hit@5","R@5"]]
print(view.sort_values("dim", ascending=False).to_string(index=False))



=== 2B.8: 8B усечённый до dim 2048 ===
  [выгружено] свободно VRAM: 23.4 ГБ
  [загрузка] Qwen/Qwen3-Embedding-8B (qwen) ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [готово] за 22 с | VRAM занято: 8.2 ГБ


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  [выгружено] свободно VRAM: 23.4 ГБ
  ✅ R@5=1.0 NDCG@5=0.9283 Hit@5=1.0 MRR=0.9034

=== 2B.9: 8B усечённый до dim 1536 ===
  [выгружено] свободно VRAM: 23.4 ГБ
  [загрузка] Qwen/Qwen3-Embedding-8B (qwen) ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [готово] за 22 с | VRAM занято: 8.2 ГБ
  [выгружено] свободно VRAM: 23.4 ГБ
  ✅ R@5=1.0 NDCG@5=0.9273 Hit@5=1.0 MRR=0.9023

=== MRL: деградация 8B при усечении ===
 run  dim  NDCG@5    MRR  Hit@5  R@5
2B.3 4096  0.9367 0.9148    1.0  1.0
2B.8 2048  0.9283 0.9034    1.0  1.0
2B.9 1536  0.9273 0.9023    1.0  1.0


In [ ]:
#@title Ячейка 9.1 — MRL для 4B (2048/1536) + статтест 4B@2048 vs 8B@2048
import pandas as pd, numpy as np, time, json, os
from scipy.stats import wilcoxon

# --- досчёт 4B на усечённых размерностях (режем кэш 4B/2560) ---
SRC_4B = ("2B.1", "Qwen/Qwen3-Embedding-4B", "qwen", 2560, "none")

mrl4b_rows = []
for tgt, rid in [(2048, "2B.8b"), (1536, "2B.9b")]:
    print(f"\n=== {rid}: 4B усечённый до dim {tgt} ===")
    try:
        row = eval_truncated(SRC_4B, tgt, rid, real_docs, QA_REAL)
        mrl4b_rows.append(row)
        print(f"  ✅ R@5={row['R@5']} NDCG@5={row['NDCG@5']} Hit@5={row['Hit@5']} MRR={row['MRR']}")
    except Exception as e:
        import traceback; traceback.print_exc(); unload_model()

# дозапись в общий CSV
df_all = pd.concat([pd.read_csv(f"{BASE}/results/phase2b_models.csv"),
                    pd.DataFrame(mrl4b_rows)], ignore_index=True)
df_all.to_csv(f"{BASE}/results/phase2b_models.csv", index=False)

# --- per-question MRR для статтеста 4B@2048 vs 8B@2048 ---
def per_q_mrr_truncated(src_run, target_dim, docs, qa_set):
    _, hf_name, family, full_dim, prompt = src_run
    rag_common.MODEL = f"{hf_name}|{prompt}"
    key = rag_common.index_key([d["id"] for d in docs], "fixed_512", 1024, 0.1,
                               rag_common.MODEL, full_dim, "q8")
    data = np.load(f"{rag_common.INDEX_CACHE}/{key}.npz", allow_pickle=True)
    meta_j = json.load(open(f"{rag_common.INDEX_CACHE}/{key}.json", encoding="utf-8"))
    sub = data["vectors"][:, :target_dim].astype(np.float32)
    sub = sub / (np.linalg.norm(sub, axis=1, keepdims=True) + 1e-12)
    idx = {"key": f"{key}_mrl{target_dim}", "vectors": sub,
           "chunks": meta_j["chunks"], "chunk_meta": meta_j["chunk_meta"]}
    sparse = SparseIndex(idx); meta = idx["chunk_meta"]
    load_model(hf_name, family); rag_common.encode = ENCODERS[family]
    CURRENT_PROMPT["mode"] = prompt
    mrrs = []
    for q in qa_set:
        rel = {q["doc_id"]: 3}
        pos = rag_common.run_search("S6", q["query"], idx, sparse_index=sparse,
                                    top_k=10, dim=target_dim, by="chunk")
        dr, seen = [], set()
        for p in pos:
            if isinstance(p, int) and p < len(meta):
                d = meta[p]["doc_id"]
                if d not in seen: seen.add(d); dr.append(d)
        mrrs.append(rag_common.mrr(dr, rel))
    unload_model()
    return np.array(mrrs)

s_4b = per_q_mrr_truncated(SRC_4B, 2048, real_docs, QA_REAL)
s_8b = per_q_mrr_truncated(("2B.3","Qwen/Qwen3-Embedding-8B","qwen",4096,"none"), 2048, real_docs, QA_REAL)

print("\n=== 4B@2048 vs 8B@2048 (целевая размерность для БД) ===")
print(f"  MRR 4B@2048: {s_4b.mean():.4f}")
print(f"  MRR 8B@2048: {s_8b.mean():.4f}")
diff = s_8b - s_4b; nz = np.sum(diff != 0)
if nz == 0:
    print("  идентичны (0 различий)")
else:
    stat, p = wilcoxon(s_8b, s_4b)
    print(f"  Wilcoxon p={p:.4f} ({'ЗНАЧИМО' if p<0.05 else 'не значимо'}), различий={nz}")

# сводка всех MRL-вариантов
print("\n=== Сводка MRL (4B и 8B на 2048/1536) ===")
full = pd.read_csv(f"{BASE}/results/phase2b_models.csv")
print(full[full["run"].isin(["2B.1","2B.3","2B.8","2B.9","2B.8b","2B.9b"])]
      [["run","model","dim","NDCG@5","MRR","Hit@5","R@5"]].to_string(index=False))



=== 2B.8b: 4B усечённый до dim 2048 ===
  [выгружено] свободно VRAM: 23.4 ГБ
  [загрузка] Qwen/Qwen3-Embedding-4B (qwen) ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [готово] за 12 с | VRAM занято: 4.4 ГБ


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  [выгружено] свободно VRAM: 23.4 ГБ
  ✅ R@5=1.0 NDCG@5=0.9224 Hit@5=1.0 MRR=0.8958

=== 2B.9b: 4B усечённый до dim 1536 ===
  [выгружено] свободно VRAM: 23.4 ГБ
  [загрузка] Qwen/Qwen3-Embedding-4B (qwen) ...


...

  [готово] за 12 с | VRAM занято: 4.4 ГБ


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  [выгружено] свободно VRAM: 23.4 ГБ
  [выгружено] свободно VRAM: 23.4 ГБ
  [загрузка] Qwen/Qwen3-Embedding-8B (qwen) ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [готово] за 22 с | VRAM занято: 8.2 ГБ
  [выгружено] свободно VRAM: 23.4 ГБ

=== 4B@2048 vs 8B@2048 (целевая размерность для БД) ===
  MRR 4B@2048: 0.8958
  MRR 8B@2048: 0.9034
  Wilcoxon p=0.5930 (не значимо), различий=3

=== Сводка MRL (4B и 8B на 2048/1536) ===
  run              model  dim  NDCG@5    MRR  Hit@5  R@5
 2B.1 Qwen3-Embedding-4B 2560  0.9254 0.8996    1.0  1.0
 2B.3 Qwen3-Embedding-8B 4096  0.9367 0.9148    1.0  1.0
 2B.8 Qwen3-Embedding-8B 2048  0.9283 0.9034    1.0  1.0
 2B.9 Qwen3-Embedding-8B 1536  0.9273 0.9023    1.0  1.0
2B.8b Qwen3-Embedding-4B 2048  0.9224 0.8958    1.0  1.0
2B.9b Qwen3-Embedding-4B 1536  0.9224 0.8958    1.0  1.0
